# 📓 Média Truncada

> **Meu Caderno Digital · Estatística e Probabilidade para IA e Dados**
> Módulo 01 · Estatística Descritiva › Medidas de Posição › Aula 04

### 🎯 Objetivo da aula:

1. Definir a média truncada e a winsorizada, e explicar a média truncada como um **botão** que vai da média (α = 0) até a mediana (α → 0,5).
2. Calculá-la **na mão** e implementá-la **do zero**, inclusive sem ordenar tudo.
3. Mostrar que seu **ponto de ruptura é α** e medir o **preço da robustez** (eficiência) para escolher α.
4. Calcular o **erro padrão** e o intervalo de confiança da média truncada (e por que o desvio padrão do "miolo" engana).
5. Reconhecer as armadilhas: convenções de α, amostras pequenas, viés em dados assimétricas, `NaN` e erro padrão ingênuo.
6. Aplicar a média truncada em **Análise de Dados, Ciência de Dados, ML, DL e Engenharia**, incluindo a **média móvel truncada** para sensores.

### 🗺️ Roteiro da aula

| Ato | Seções |
|---|---|
| **I · Por quê** | 1. O problema · 2. A intuição |
| **II · O quê** | 3. Definição formal · 4. Propriedades e relações |
| **III · Como** | 5. Na mão · 6. Em Python · 7. Visualizando |
| **IV · Cuidado** | 8. Armadilhas e limites |
| **V · Onde se aplica** | 9. No trabalho, por área · 10. Mini-caso |
| **VI · Fixar** | 11. Entrevista e exercícios · 12. Resumo e ponte |

> **Legenda dos blocos:** 🧠 Intuição · ⚠️ Armadilha · 🏭 No trabalho · 🎯 Entrevista

In [ ]:
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import mstats
from numpy.lib.stride_tricks import sliding_window_view

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

plt.rcParams.update({
    "figure.dpi": 100, "figure.figsize": (9, 4.2), "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
})
COR = {"media": "#d1495b", "mediana": "#00798c", "trunc": "#6a4c93", "base": "#30638e", "ponderada": "#edae49"}
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

# O dataset fica em data/, dois níveis acima deste notebook
df = pd.read_csv("../../data/dataset_estatistica.csv")
renda = df["renda_mensal"].dropna()
r = renda.to_numpy()
print(f"Dataset carregado: {df.shape[0]} linhas x {df.shape[1]} colunas | renda válida: {len(r)} valores")

---
# ATO I · POR QUÊ
## 1 · O problema: o jurado que destruiu a nota

Um saltador de trampolim é avaliado por 7 jurados. Seis deles dão notas parecidas, e um (talvez por erro de digitação, talvez por parcialidade) dá **3,0**:

In [ ]:
notas = np.array([8.5, 8.0, 8.5, 9.0, 8.0, 8.5, 3.0])
ordenadas = np.sort(notas)
miolo = ordenadas[1:-1]                                      # descarta a menor e a maior nota

print("notas ordenadas :", ordenadas)
print(f"média simples   = {notas.mean():.2f}")
print(f"mediana         = {np.median(notas):.2f}   (usa só 1 das {len(notas)} notas)")
print(f"média do miolo  = {miolo.mean():.2f}   (descarta a maior e a menor; usa {len(miolo)} das {len(notas)} notas)")

Um único jurado puxou a **média** para baixo, o que é injusto com o atleta. A **mediana** resolve, mas joga fora quase toda a informação: com 7 jurados, ela só olha para a nota do meio. A terceira linha é um meio-termo natural: **descartar só as pontas** e fazer a média do resto. É o que se faz em muitos esportes julgados por painel, e é o que a estatística chama de **média truncada** (ou *aparada*).

Essa ideia levanta duas perguntas que guiam a aula:

1. Quantas notas devo descartar de cada lado? Existe um critério, ou é "chute"?
2. O que eu **ganho** e o que eu **perco**, em relação à média e à mediana?

---
## 2 · A intuição: corte as pontas e faça a média do miolo

> 🧠 **Intuição 1 · Aparar as pontas.** Ordene os valores e **descarte uma fração das duas pontas**, onde moram os outliers. Depois faça a média simples do que sobrou. É como podar uma árvore: tira o que está fora do lugar e mantém o tronco.

> 🧠 **Intuição 2 · Um botão entre a média e a mediana.** A fração cortada (α) é um **botão**. Com α = 0 não se corta nada, e você tem a **média**. Aumentando α, você confia cada vez menos nas pontas. Quando α chega perto de 0,5, só sobra o valor do meio, e você tem a **mediana**. Escolher α é escolher **o quanto você desconfia dos dados**.

> 🧠 **Intuição 3 · Informação do miolo.** A mediana usa uma ou duas observações. A média truncada usa **a maior parte** delas, por isso costuma ser mais precisa que a mediana quando os dados são razoavelmente limpos.

Veja os dois retratos da ideia: à esquerda, os valores ordenados com as pontas cortadas (cinza); à direita, o **botão α** aplicado à renda, indo da média até a mediana:

In [ ]:
amostra = np.sort(np.round(rng.lognormal(3.0, 0.5, 20), 1))
k = int(0.10 * len(amostra))                                  # 10% em cada ponta = 2 valores de cada lado
cores = [("#bbbbbb" if (i < k or i >= len(amostra) - k) else COR["base"]) for i in range(len(amostra))]

fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
axs[0].bar(range(1, len(amostra) + 1), amostra, color=cores)
axs[0].axhline(amostra.mean(), color=COR["media"], ls="--", lw=2, label=f"média = {amostra.mean():.1f}")
axs[0].axhline(amostra[k:-k].mean(), color=COR["trunc"], lw=2.5, label=f"truncada 10% = {amostra[k:-k].mean():.1f}")
axs[0].set_xlabel("posição (valores ordenados; cinza = cortados)"); axs[0].set_ylabel("valor")
axs[0].set_title("Cortar as pontas e fazer a média do miolo"); axs[0].legend(loc="upper left")

alfas = np.linspace(0, 0.49, 50)
curva = np.array([stats.trim_mean(r, a) for a in alfas])
axs[1].plot(alfas, curva, color=COR["trunc"], lw=2.5)
axs[1].axhline(r.mean(), color=COR["media"], ls="--", label=f"média (α = 0): {r.mean():.0f}")
axs[1].axhline(np.median(r), color=COR["mediana"], ls=":", lw=2.5, label=f"mediana (α → 0,5): {np.median(r):.0f}")
axs[1].set_xlabel("fração cortada em CADA ponta (α)"); axs[1].set_ylabel("renda estimada (R$)")
axs[1].set_title("O botão α: da média até a mediana"); axs[1].legend()
plt.tight_layout(); plt.show()

---
# ATO II · O QUÊ
## 3 · Definição formal

**Média truncada** (ou *aparada*) com corte $\alpha$. Ordene os valores, $x_{(1)} \le \dots \le x_{(n)}$, descarte $k = \lfloor \alpha n \rfloor$ valores de **cada** ponta e tire a média dos $n - 2k$ restantes:

$$\bar{x}_{\alpha} = \frac{1}{n - 2k}\sum_{i=k+1}^{n-k} x_{(i)}, \qquad k = \lfloor \alpha n \rfloor, \quad 0 \le \alpha < \tfrac{1}{2}$$

Fala-se em "média truncada de 10%" para dizer que $\alpha = 0{,}10$ em **cada ponta** (20% dos dados descartados no total).

**Média winsorizada.** Em vez de **descartar** os extremos, ela os **substitui** pelo valor vizinho mais próximo que foi mantido, preservando o tamanho $n$:

$$\bar{x}_{w} = \frac{1}{n}\Big[\,k\,x_{(k+1)} + \sum_{i=k+1}^{n-k} x_{(i)} + k\,x_{(n-k)}\Big]$$

**Versão populacional.** Em termos da função quantil $Q(u) = F^{-1}(u)$, a média truncada estima

$$\mu_{\alpha} = \frac{1}{1 - 2\alpha}\int_{\alpha}^{1-\alpha} Q(u)\,du$$

| Símbolo | Nome | O que é |
|---|---|---|
| $\alpha$ | fração de corte | proporção descartada em cada ponta |
| $k$ | nº de valores descartados por ponta | $\lfloor \alpha n \rfloor$ |
| $\bar{x}_\alpha$ | média truncada | média do miolo |
| $\bar{x}_w$ | média winsorizada | média após "empurrar" os extremos para dentro |

### Quatro conexões que valem ouro

1. **A família da média até a mediana.** Com $\alpha = 0$ ($k=0$) é a **média**. Quando $\alpha$ se aproxima de $0{,}5$, sobra só o centro: a **mediana**. Entre os dois extremos, uma família inteira de estimadores.
2. **É um estimador L.** Média, mediana, média truncada e winsorizada são **combinações lineares das estatísticas de ordem**, $\sum c_i\, x_{(i)}$, só mudam os pesos $c_i$.
3. **Ponto de ruptura $= \alpha$.** Já vimos que a média quebra com 0% de contaminação e a mediana com 50%. A truncada tolera até a fração $\alpha$ (seção 4, P2).
4. **Substitui um critério subjetivo.** "Remover outliers e depois fazer a média" exige uma regra do analista. A truncada tem uma regra **mecânica** e simétrica: corta por **posição**, e não por valor.

---
## 4 · Propriedades e relações

Vamos **verificar cada propriedade com código**.

### P1 · A média truncada interpola entre a média e a mediana

In [ ]:
print(f"α = 0       -> {stats.trim_mean(r, 0.0):.4f}  (média   = {r.mean():.4f})")
print(f"α = 0,4999  -> {stats.trim_mean(r, 0.4999):.4f}  (mediana = {np.median(r):.4f})")
assert np.isclose(stats.trim_mean(r, 0.0), r.mean())
assert np.isclose(stats.trim_mean(r, 0.4999), np.median(r))       # n ímpar: sobra só o valor central

### P2 · Ponto de ruptura = α

Contaminamos uma fração crescente da amostra (101 valores) com o valor absurdo $10^6$ e observamos quando cada estimador "quebra":

In [ ]:
g = np.random.default_rng(21)
base = g.normal(50, 10, 101)
estimadores = {
    "média":         lambda z: z.mean(),
    "truncada 10%":  lambda z: stats.trim_mean(z, 0.10),
    "truncada 25%":  lambda z: stats.trim_mean(z, 0.25),
    "mediana":       np.median,
}
ms = np.arange(0, 56)
resposta = {nome: [] for nome in estimadores}
for m in ms:
    z = base.copy(); z[:m] = 1e6
    for nome, f in estimadores.items():
        resposta[nome].append(f(z))

fig, ax = plt.subplots(figsize=(9.5, 4.4))
cores = {"média": COR["media"], "truncada 10%": "#b388eb", "truncada 25%": COR["trunc"], "mediana": COR["mediana"]}
for nome, v in resposta.items():
    ax.semilogy(ms / len(base), v, color=cores[nome], lw=2.2, label=nome)
for a_, cor in [(0.10, "#b388eb"), (0.25, COR["trunc"]), (0.50, COR["mediana"])]:
    ax.axvline(a_, color=cor, ls=":", lw=1.3)
ax.set_xlabel("fração de valores contaminados (10⁶)"); ax.set_ylabel("estimativa (escala log)")
ax.set_title("Cada estimador quebra exatamente na sua fração de corte"); ax.legend(loc="center left")
plt.tight_layout(); plt.show()

assert resposta["truncada 10%"][10] < 200 and resposta["truncada 10%"][11] > 1e3      # k = 10 valores por ponta
assert resposta["truncada 25%"][25] < 200 and resposta["truncada 25%"][26] > 1e3      # k = 25
print(f"truncada 10%: com 10 contaminados = {resposta['truncada 10%'][10]:.1f} | com 11 = {resposta['truncada 10%'][11]:.0f}")
print(f"truncada 25%: com 25 contaminados = {resposta['truncada 25%'][25]:.1f} | com 26 = {resposta['truncada 25%'][26]:.0f}")

Cada curva fica **estável** até a sua fração de corte e **explode** na primeira observação além dela. Isso mostra o **trade-off** em ação: a média truncada de 10% é protegida contra até 10% de contaminação, e a de 25%, contra até 25%.

### P3 · O preço da robustez: eficiência estatística por α

Na Aula 02 vimos que a mediana, em dados limpos, é só cerca de 64% tão precisa quanto a média. Prometemos um meio-termo. Vamos medi-lo: para cada α, a razão $\text{Var}(\bar{x}) / \text{Var}(\bar{x}_\alpha)$ em milhares de amostras de 101 valores. **Acima de 1**, a truncada é mais precisa que a média; **abaixo de 1**, é menos.

In [ ]:
def normal_contaminada(g, shape):                    # 90% N(0,1) + 10% N(0,10)
    base_ = g.normal(0, 1, shape)
    ruim = g.random(shape) < 0.10
    return np.where(ruim, g.normal(0, 10, shape), base_)

cenarios = {
    "Uniforme":                lambda g, s: g.uniform(-1, 1, s),
    "Normal":                  lambda g, s: g.normal(0, 1, s),
    "t de Student (3 g.l.)":   lambda g, s: g.standard_t(3, s),
    "Laplace":                 lambda g, s: g.laplace(0, 1, s),
    "Normal contaminada":      normal_contaminada,
}
alfas_tab = [0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.49]
n_amostra, reps = 101, 6000
g = np.random.default_rng(0)

eficiencia = {}
for nome, gerar in cenarios.items():
    amostras = np.sort(gerar(g, (reps, n_amostra)), axis=1)
    var_media = amostras.mean(axis=1).var()
    linha = []
    for a_ in alfas_tab:
        k_ = int(a_ * n_amostra)
        linha.append(var_media / amostras[:, k_:n_amostra - k_].mean(axis=1).var())
    eficiencia[nome] = linha
tab = pd.DataFrame(eficiencia, index=[f"{a_:.0%}" for a_ in alfas_tab])
tab.index.name = "corte α"
display(tab.round(3))

assert tab.loc["10%", "Normal"] > 0.90                      # cortar 10% quase não custa nada em dados normais
assert tab.loc["10%", "Normal contaminada"] > 1.5           # e protege muito quando há contaminação
assert tab.loc["49%", "Normal"] < tab.loc["10%", "Normal"]  # cortar demais (≈ mediana) custa precisão

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.4))
for nome in tab.columns:
    ax.plot(alfas_tab, tab[nome].values, marker="o", label=nome)
ax.axhline(1, color="#666666", ls=":")
ax.set_xlabel("corte α (por ponta)"); ax.set_ylabel("Var(média) / Var(truncada)   (> 1: a truncada é melhor)")
ax.set_title("Eficiência da média truncada em relação à média, por tipo de dado"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# Qual α é um bom compromisso? Compare cada α com o MELHOR α de cada cenário e olhe o pior caso.
relativa = tab / tab.max(axis=0)
pior_caso = relativa.min(axis=1)
print("Eficiência relativa ao melhor α de cada cenário, no pior cenário:")
display(pior_caso.round(2).to_frame("pior caso"))
print(f"O corte que maximiza o pior caso é α = {pior_caso.idxmax()}.")

A leitura é direta:

- Em **dados normais**, cortar 10% custa muito pouco, e cortar 49% (quase a mediana) custa bastante.
- Com **caudas pesadas ou contaminação**, a truncada **supera a média** com folga, e o ganho é grande já com cortes pequenos (5 a 10%).
- O corte do "pior caso" (o que menos perde, no pior cenário, em relação ao melhor corte de cada cenário) foi de **10%** neste experimento. Por isso valores como 10%, 15% e 20% são tão populares: são o compromisso que não vai muito mal em nenhum cenário.

### P4 · Truncar vs. winsorizar

A winsorização **troca** os extremos pelos valores vizinhos em vez de descartá-los. Mantém $n$ e costuma ser usada para calcular **variâncias** e erros padrão (P6).

In [ ]:
from scipy.stats.mstats import winsorize

def media_winsorizada(x, alfa):
    s = np.sort(np.asarray(x, dtype=float))
    n = len(s); k = int(alfa * n)
    if k > 0:
        s[:k] = s[k]                        # a ponta inferior vira o menor valor mantido
        s[n - k:] = s[n - k - 1]            # a ponta superior vira o maior valor mantido
    return s.mean()

alfa = 0.10
assert np.isclose(media_winsorizada(r, alfa), float(winsorize(r, limits=(alfa, alfa)).mean()))
comparacao = pd.Series({
    "média": r.mean(),
    "truncada 10%": stats.trim_mean(r, alfa),
    "winsorizada 10%": media_winsorizada(r, alfa),
    "mediana": np.median(r),
}, name="renda (R$)")
comparacao.round(2).to_frame()

### P5 · Equivariante a transformações afins, mas **não aditiva**

Como a mediana, a média truncada acompanha mudanças de escala e de unidade ($\bar{x}_\alpha(ax + b) = a\,\bar{x}_\alpha(x) + b$), mas **a truncada da soma não é a soma das truncadas**, porque os cortes dependem da **ordem** de cada variável:

In [ ]:
assert np.isclose(stats.trim_mean(0.18 * r + 100, 0.10), 0.18 * stats.trim_mean(r, 0.10) + 100)
assert np.isclose(stats.trim_mean(-r, 0.10), -stats.trim_mean(r, 0.10))
print("equivariância afim: confere (inclusive com sinal negativo).")

x1 = np.array([1, 2, 10, 50, 100]); y1 = np.array([100, 50, 2, 10, 1])
soma_das = stats.trim_mean(x1, 0.2) + stats.trim_mean(y1, 0.2)
da_soma = stats.trim_mean(x1 + y1, 0.2)
print(f"truncada(x) + truncada(y) = {soma_das:.2f}   |   truncada(x + y) = {da_soma:.2f}   <- diferentes")

### P6 · Erro padrão e intervalo de confiança

> ⚠️ **O desvio padrão dos valores que sobraram NÃO serve** como base do erro padrão. Depois de cortar, o miolo ficou artificialmente "arrumado", e o desvio dele **subestima** a incerteza. A fórmula de **Tukey e McLaughlin** usa o desvio padrão da amostra **winsorizada**:

$$\text{EP}(\bar{x}_\alpha) = \frac{s_w}{(1 - 2\alpha)\sqrt{n}} \qquad \text{IC} = \bar{x}_\alpha \pm t_{n-2k-1,\;0{,}975}\cdot \text{EP}$$

In [ ]:
def erro_padrao_truncada(x, alfa):
    s = np.sort(np.asarray(x, dtype=float))
    n = len(s); k = int(alfa * n)
    w = s.copy()
    w[:k] = s[k]; w[n - k:] = s[n - k - 1]                # amostra winsorizada
    return w.std(ddof=1) / ((1 - 2 * k / n) * np.sqrt(n))

g = np.random.default_rng(11)
boot = np.array([stats.trim_mean(g.choice(r, size=len(r), replace=True), 0.10) for _ in range(2000)])
ep_form = erro_padrao_truncada(r, 0.10)
ep_boot = boot.std(ddof=1)
print(f"erro padrão (fórmula de Tukey–McLaughlin) = {ep_form:.2f}")
print(f"erro padrão (bootstrap, 2000 reamostras)  = {ep_boot:.2f}")
assert 0.7 < ep_form / ep_boot < 1.3

n_, k_ = len(r), int(0.10 * len(r))
tcrit = stats.t.ppf(0.975, n_ - 2 * k_ - 1)
tm = stats.trim_mean(r, 0.10)
print(f"\nIC 95% para a média truncada de 10% da renda: [{tm - tcrit * ep_form:.0f}, {tm + tcrit * ep_form:.0f}]  (estimativa pontual = {tm:.0f})")

A fórmula e o *bootstrap* concordam, e é por isso que a média truncada é um estimador **inferencialmente utilizável**: tem erro padrão, intervalo e até teste de hipótese (seção 9.2).

### Quatro medidas de posição lado a lado

| | Média | Truncada (α) | Mediana |
|---|---|---|---|
| **Usa** | todos os valores | o miolo ($n - 2k$ valores) | 1 ou 2 valores centrais |
| **Ponto de ruptura** | 0% | **α** | 50% |
| **Eficiência na Normal** | 100% | alta (≈ 94% em α = 10%, tipicamente) | ≈ 64% |
| **Com caudas pesadas** | frágil | **boa** | boa |
| **Erro padrão** | $s/\sqrt{n}$ | winsorizado (P6) | mais complexo |
| **Estima** | $\mu$ | $\mu_\alpha$ (**≠ $\mu$** se assimétrica) | mediana populacional |
| **Parâmetro a escolher** | nenhum | **α** | nenhum |

---
# ATO III · COMO
## 5 · Na mão

**Exemplo.** Dez notas: `[9, 7, 8, 2, 8, 10, 9, 7, 8, 9]`. Primeiro, **ordene**: `[2, 7, 7, 8, 8, 8, 9, 9, 9, 10]`. A soma é 77, logo a média simples é $77/10 = 7{,}7$ (puxada pelo 2).

| α | $k = \lfloor \alpha n \rfloor$ | O que sobra | Média do miolo |
|---|---|---|---|
| 0 | 0 | as 10 notas | $77/10 = 7{,}70$ |
| 0,1 | 1 | `[7, 7, 8, 8, 8, 9, 9, 9]` | $65/8 = 8{,}125$ |
| 0,2 | 2 | `[7, 8, 8, 8, 9, 9]` | $49/6 \approx 8{,}167$ |
| 0,3 | 3 | `[8, 8, 8, 9]` | $33/4 = 8{,}25$ |
| 0,4 | 4 | `[8, 8]` | $16/2 = 8{,}00$ (a **mediana**) |

Repare que a média sobe de 7,7 para ~8,1 assim que o "2" é descartado, e que com $\alpha = 0{,}4$ a média truncada **vira a mediana**.

Agora em Python **puro**, com as validações:

In [ ]:
def media_truncada(x, alfa):
    """Média truncada: descarta k = floor(alfa * n) valores de CADA ponta e tira a média do miolo."""
    n = len(x)
    if n == 0:
        raise ValueError("a lista está vazia")
    if not 0 <= alfa < 0.5:
        raise ValueError("alfa precisa estar em [0, 0.5)")
    k = int(alfa * n)
    miolo = sorted(x)[k:n - k]
    return sum(miolo) / len(miolo)


notas10 = [9, 7, 8, 2, 8, 10, 9, 7, 8, 9]
for alfa in [0, 0.1, 0.2, 0.3, 0.4]:
    print(f"α = {alfa:.1f} (k = {int(alfa * len(notas10))}) -> média truncada = {media_truncada(notas10, alfa):.4f}")

In [ ]:
# Testes: precisamos concordar com o SciPy e falhar com clareza nos casos de borda
assert np.isclose(media_truncada(notas10, 0.0), np.mean(notas10))
assert np.isclose(media_truncada(notas10, 0.4), np.median(notas10))
for _ in range(300):
    amostra = rng.normal(50, 15, size=rng.integers(5, 80))
    a_ = rng.uniform(0, 0.49)
    assert np.isclose(media_truncada(amostra.tolist(), a_), stats.trim_mean(amostra, a_))

for chamada, descricao in [(lambda: media_truncada([], 0.1), "lista vazia"),
                           (lambda: media_truncada([1, 2, 3], 0.5), "alfa = 0,5"),
                           (lambda: media_truncada([1, 2, 3], -0.1), "alfa negativo")]:
    try:
        chamada()
    except ValueError as erro:
        print(f"OK  {descricao:<14} -> ValueError: {erro}")

### Bônus de engenharia: truncar sem ordenar tudo

Como na mediana, não precisamos **ordenar** o vetor inteiro, só **separar** os $k$ menores, o miolo e os $k$ maiores. O `np.partition` faz isso em tempo médio linear:

In [ ]:
def media_truncada_rapida(x, alfa):
    x = np.asarray(x, dtype=float)
    n = len(x); k = int(alfa * n)
    if k == 0:
        return x.mean()
    p = np.partition(x, [k, n - k - 1])        # posiciona os cortes sem ordenar o resto
    return p[k:n - k].mean()                   # o miolo (desordenado) está entre as posições k e n-k-1


grande = rng.normal(size=2_000_000)
assert np.isclose(media_truncada_rapida(grande, 0.10), stats.trim_mean(grande, 0.10))

def melhor_tempo(f, repeticoes=5):
    tempos = []
    for _ in range(repeticoes):
        t0 = time.perf_counter(); f(); tempos.append(time.perf_counter() - t0)
    return min(tempos)

t_ordenar = melhor_tempo(lambda: np.sort(grande)[int(0.1 * len(grande)):len(grande) - int(0.1 * len(grande))].mean())
t_particao = melhor_tempo(lambda: media_truncada_rapida(grande, 0.10))
print(f"ordenando tudo     : {t_ordenar * 1000:7.1f} ms")
print(f"com np.partition   : {t_particao * 1000:7.1f} ms   ({t_ordenar / t_particao:.1f}x mais rápido)")

---
## 6 · Em Python: SciPy e Pandas

O NumPy e o Pandas **não têm** média truncada embutida. A ferramenta padrão é o `scipy.stats.trim_mean`, que recebe a fração cortada **em cada ponta**:

In [ ]:
r_com_nan = df["renda_mensal"].to_numpy()                         # 11 valores ausentes

print(f"scipy.stats.trim_mean (sem NaN)        : {stats.trim_mean(r, 0.10):.2f}")
print(f"trimboth + média                       : {stats.trimboth(np.sort(r), 0.10).mean():.2f}")
print(f"mstats.trimmed_mean (as duas pontas)   : {float(mstats.trimmed_mean(r, limits=(0.10, 0.10))):.2f}")
print(f"mstats.trimmed_mean (só a ponta direita): {float(mstats.trimmed_mean(r, limits=(None, 0.10))):.2f}")
print(f"\nscipy.stats.trim_mean (COM NaN)        : {stats.trim_mean(r_com_nan, 0.10)}")

> ⚠️ **Atenção aos `NaN`.** Nenhuma dessas funções trata os ausentes por você. Filtre **antes** (`dropna`), e veja na seção 8 por que isso é mais traiçoeiro do que parece.

O `mstats.trimmed_mean` também aceita **corte assimétrico** (`limits=(None, 0.10)` corta só a ponta direita), útil quando os outliers só existem de um lado, como em renda ou tempos de espera.

### Média truncada por grupo

Para usar em `groupby`, basta embrulhar o `trim_mean` em uma função. A tabela abaixo é o "relatório robusto" da renda por categoria:

In [ ]:
def aparar(alfa):
    return lambda s: stats.trim_mean(s.dropna(), alfa)

resumo = df.groupby("categoria_frequente", observed=True)["renda_mensal"].agg(
    n="count", media="mean", trunc10=aparar(0.10), trunc20=aparar(0.20), mediana="median",
)
resumo["media − trunc10"] = resumo["media"] - resumo["trunc10"]
resumo.round(1)

A coluna `media − trunc10` mostra **quanto a média está sendo puxada pelas pontas** em cada categoria, um diagnóstico útil de assimetria e outliers.

---
## 7 · Visualizando

Duas vistas: o **botão α** por categoria e o que ele faz com a **incerteza**, medida por *bootstrap* (1000 reamostragens da renda):

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.4))

# (a) o botão α em cada categoria
for categoria, s in df.groupby("categoria_frequente", observed=True)["renda_mensal"]:
    v = s.dropna().to_numpy()
    axs[0].plot(alfas, [stats.trim_mean(v, a_) for a_ in alfas], lw=2, label=f"{categoria} (n = {len(v)})")
axs[0].set_xlabel("fração cortada em cada ponta (α)"); axs[0].set_ylabel("renda estimada (R$)")
axs[0].set_title("A truncada vai da média (α = 0) à mediana (α → 0,5)"); axs[0].legend(fontsize=9)

# (b) a distribuição dos estimadores por bootstrap
g = np.random.default_rng(3)
reamostras = [g.choice(r, size=len(r), replace=True) for _ in range(1000)]
est = {
    "média":        [v.mean() for v in reamostras],
    "truncada 10%": [stats.trim_mean(v, 0.10) for v in reamostras],
    "truncada 25%": [stats.trim_mean(v, 0.25) for v in reamostras],
    "mediana":      [np.median(v) for v in reamostras],
}
axs[1].boxplot(list(est.values()), showmeans=True,
               medianprops={"color": "black"},
               meanprops={"marker": "D", "markerfacecolor": COR["media"], "markeredgecolor": "white"})
axs[1].set_xticks(range(1, len(est) + 1)); axs[1].set_xticklabels(est.keys())
axs[1].set_ylabel("estimativa da renda (R$)"); axs[1].set_title("Bootstrap: onde cada estimador cai")
plt.tight_layout(); plt.show()

for nome, v in est.items():
    print(f"{nome:<14} média das reamostras = {np.mean(v):7.1f} | desvio padrão = {np.std(v, ddof=1):6.1f}")

Dois fatos saltam do gráfico (b): a **posição** de cada estimador desce da média até a mediana (cada corte tira peso da cauda direita), e a **largura** das caixas (o erro padrão) mostra a precisão de cada um.

---
# ATO IV · CUIDADO
## 8 · Armadilhas e limites

A média truncada parece uma "média robusta sem custo", e por isso é fácil usá-la mal. Cinco armadilhas.

### 8.1 · Convenções de α diferentes

> ⚠️ "Truncar 20%" **não** significa a mesma coisa em todo lugar. No SciPy e no R (`mean(x, trim=0.2)`), $\alpha$ é a fração **por ponta**. No Excel (`TRIMMEAN(dados, 0.2)`), o argumento é a fração **total** descartada, dividida entre as duas pontas.

In [ ]:
def trimmean_excel(x, p):
    """Convenção do Excel: p é a fração TOTAL descartada (soma das duas pontas), com arredondamento para baixo."""
    s = np.sort(np.asarray(x, dtype=float)); n = len(s); k = int(p * n / 2)
    return s[k:n - k].mean()

g = np.random.default_rng(9)
x20 = np.sort(g.normal(50, 10, 20)); x20[-1] = 500                   # um outlier
print(f"SciPy  trim_mean(x, 0.20)  -> descarta k = {int(0.2 * 20)} por ponta | resultado = {stats.trim_mean(x20, 0.20):.2f}")
print(f"Excel  TRIMMEAN(x, 0.20)   -> descarta k = {int(0.2 * 20 / 2)} por ponta | resultado = {trimmean_excel(x20, 0.20):.2f}")
print(f"média simples                                          | resultado = {x20.mean():.2f}")

O "mesmo" 0,20 descartou **4 valores** em cada ponta em uma convenção e **2** na outra. Ao reportar uma média truncada, **escreva a convenção** ("10% por ponta, 20% no total").

### 8.2 · Amostras pequenas: a truncada vira a média sem avisar

> ⚠️ Se $\alpha n < 1$, então $k = 0$ e **nada é cortado**. Quem pede "10% de truncamento" em 9 valores recebe, silenciosamente, a média comum:

In [ ]:
g = np.random.default_rng(1)
for n_ in [5, 9, 10, 19]:
    z = np.append(g.normal(50, 2, n_ - 1), 500.0)                    # um outlier em n_ valores
    k = int(0.10 * n_)
    print(f"n = {n_:>2}: k = {k} -> trim_mean(10%) = {stats.trim_mean(z, 0.10):8.2f} | média = {z.mean():8.2f}")
z9 = np.append(g.normal(50, 2, 8), 500.0)
assert np.isclose(stats.trim_mean(z9, 0.10), z9.mean())           # n = 9, k = 0: a "truncada" é a própria média
print(f"\nPara cortar ao menos um valor por ponta com n = 9, é preciso α >= 1/9 = {1 / 9:.3f}.")

Para cortar pelo menos 1 valor por ponta, é preciso $\alpha \ge 1/n$. Em amostras pequenas, **confira $k = \lfloor \alpha n \rfloor$** antes de confiar no resultado.

### 8.3 · Em dados assimétricos, ela estima **outra coisa** (e erra totais)

> ⚠️ Em distribuições simétricas, a média truncada e a média estimam o **mesmo** valor. Em distribuições **assimétricas**, não: cortar o mesmo tanto de cada lado remove mais "massa" da cauda longa, e a truncada passa a estimar um parâmetro **menor** que a média populacional.

In [ ]:
g = np.random.default_rng(5)
grande_ln = g.lognormal(0, 0.6, 1_000_000)
print(f"média verdadeira (teoria)      = {np.exp(0.6 ** 2 / 2):.4f}")
print(f"média amostral (1 milhão)      = {grande_ln.mean():.4f}")
print(f"truncada 10% (1 milhão)        = {stats.trim_mean(grande_ln, 0.10):.4f}   <- converge para outro valor")
assert stats.trim_mean(grande_ln, 0.10) < grande_ln.mean()

total_real = r.sum()
total_trunc = len(r) * stats.trim_mean(r, 0.10)
print(f"\nReceita total (soma real)       = {total_real:.0f}")
print(f"n × média truncada              = {total_trunc:.0f}   -> subestima o total em {1 - total_trunc / total_real:.1%}")

Isso **não é um defeito**, e sim uma diferença de **objetivo**: a truncada responde "qual é o valor **típico**?", e não "qual é o **total** dividido pelo número de casos?". Para **receita, custo total ou orçamento**, use a **média**, porque a soma importa. Para **valor típico**, use a truncada ou a mediana.

### 8.4 · `NaN`: o ausente vira "outlier" e o outlier de verdade sobrevive

> ⚠️ Ao ordenar, o NumPy coloca os `NaN` **no fim** do vetor. Uma implementação baseada em `np.sort` então **corta o `NaN` como se fosse o maior valor**, e o outlier de verdade, que deveria ser cortado, **permanece**:

In [ ]:
def trunc_np(x, alfa):
    s = np.sort(np.asarray(x, dtype=float)); n = len(s); k = int(alfa * n)
    return s[k:n - k].mean()

dados = np.array([10, 11, 12, 13, 14, 15, 1000, np.nan])
certo = stats.trim_mean(dados[~np.isnan(dados)], 0.15)
print(f"correto (filtrando o NaN antes)    : {certo}")
print(f"nossa função, com o NaN no vetor   : {trunc_np(dados, 0.15)}")
print(f"scipy.stats.trim_mean com o NaN    : {stats.trim_mean(dados, 0.15)}")
assert not np.isclose(trunc_np(dados, 0.15), certo)

A nossa função devolve um número que **parece razoável** (177,5), mas está **errado**: o correto é 13. O SciPy, neste caso, devolve `nan`, o que ao menos denuncia o problema, mas implementações caseiras baseadas em `np.sort` falham **em silêncio**. Sempre **filtre os ausentes antes** de truncar, e nunca assuma que a função "se vira".

### 8.5 · Erro padrão ingênuo: o intervalo fica estreito demais

> ⚠️ Calcular o erro padrão como $s/\sqrt{n}$ **com os valores que sobraram** subestima a incerteza (veja P6). Comparando com o *bootstrap*:

In [ ]:
alfa = 0.10
k = int(alfa * len(r))
miolo = np.sort(r)[k:len(r) - k]
ep_ingenuo = miolo.std(ddof=1) / np.sqrt(len(miolo))
print(f"erro padrão ingênuo (desvio do miolo)      = {ep_ingenuo:.2f}")
print(f"erro padrão correto (winsorizado)          = {ep_form:.2f}")
print(f"erro padrão por bootstrap                  = {ep_boot:.2f}")
assert ep_ingenuo < ep_boot

O erro padrão ingênuo é **menor** que o verdadeiro, então os intervalos de confiança e os testes ficam **excessivamente confiantes**. Use a fórmula winsorizada ou o *bootstrap*.

### Resumo das armadilhas

| Armadilha | Sintoma | Remédio |
|---|---|---|
| Convenção de α | "20%" corta quantidades diferentes | escrever "por ponta" ou "no total" |
| Amostra pequena | $k = 0$: nada é cortado | conferir $k = \lfloor \alpha n \rfloor \ge 1$ |
| Dados assimétricos | subestima totais e a média | usar a média para totais |
| `NaN` | o ausente é cortado e o outlier fica | filtrar antes |
| Erro padrão ingênuo | IC estreito demais | winsorizado ou bootstrap |

---
# ATO V · ONDE SE APLICA
## 9 · No trabalho, por área

### 9.1 · Análise de Dados: KPIs e índices robustos

> 🏭 **No trabalho.** A média truncada aparece quando a métrica precisa ser **estável** diante de valores extremos: ticket médio, tempo de atendimento, preço de mercado. Um exemplo clássico em economia: bancos centrais, incluindo o do Brasil, acompanham **medidas de núcleo da inflação por médias aparadas**, que descartam as variações de preços mais extremas para enxergar a tendência de fundo.

Em SQL, uma média truncada **aproximada** usa o ranking percentual:

    -- Média truncada de 10% por ponta (aproximada), em bancos com funções de janela
    SELECT AVG(renda) AS media_truncada
    FROM (
        SELECT renda, PERCENT_RANK() OVER (ORDER BY renda) AS pr
        FROM clientes
    ) t
    WHERE pr >= 0.10 AND pr <= 0.90;

A tabela `resumo` da seção 6 é o relatório completo: média, truncadas, mediana e o **n** de cada grupo.

### 9.2 · Ciência de Dados: teste A/B robusto (o teste de Yuen)

> 🏭 **No trabalho.** Métricas de receita por usuário têm **cauda longa** e "baleias" (poucos usuários que gastam muito). Um teste t comum perde **poder** com elas. O **teste de Yuen**, que compara **médias truncadas** com variâncias winsorizadas, é o equivalente robusto, e o `scipy.stats.ttest_ind` o implementa pelo argumento `trim`. Vamos comparar o poder dos dois em 400 experimentos simulados, nos quais **existe** um efeito real e 1% dos usuários são baleias (50 vezes maiores):

In [ ]:
g = np.random.default_rng(2)

def uma_rodada(n=200, desloc=0.25, baleias=0.01):
    a = g.lognormal(0.0, 1.0, n)
    b = g.lognormal(desloc, 1.0, n)
    for v in (a, b):
        v[g.random(n) < baleias] *= 50
    return a, b

p_welch, p_yuen = [], []
for _ in range(400):
    a, b = uma_rodada()
    p_welch.append(stats.ttest_ind(a, b, equal_var=False).pvalue)
    p_yuen.append(stats.ttest_ind(a, b, equal_var=False, trim=0.10).pvalue)

poder_w, poder_y = np.mean(np.array(p_welch) < 0.05), np.mean(np.array(p_yuen) < 0.05)
print(f"poder do teste t de Welch (médias)        = {poder_w:.1%}")
print(f"poder do teste de Yuen (truncada de 10%)  = {poder_y:.1%}")
assert poder_y > poder_w

O teste de Yuen **detecta o efeito real com muito mais frequência**, porque as baleias deixam de dominar a variância. O preço, como na seção 8.3, é que ele compara os valores **típicos**, e não os totais. Para decisões de **receita total**, é preciso olhar a média também.

### 9.3 · Machine Learning: regressão truncada (LTS) e ensembles robustos

> 🏭 **No trabalho.** Aplicar a ideia de "cortar as pontas" **às perdas** dá a **regressão dos mínimos quadrados truncados** (*Least Trimmed Squares*, LTS): em vez de minimizar a soma de **todos** os erros quadráticos, minimiza-se a soma dos **menores** $h$ (por exemplo, 80%), ignorando os piores. Um algoritmo simples alterna entre ajustar e descartar os maiores resíduos:

In [ ]:
from sklearn.linear_model import LinearRegression

def regressao_truncada(X, y, fracao=0.8, iteracoes=20):
    h = int(fracao * len(y))
    modelo = LinearRegression().fit(X, y)
    for _ in range(iteracoes):
        residuos = (y - modelo.predict(X)) ** 2
        manter = np.argsort(residuos)[:h]                      # só os h menores erros decidem
        modelo = LinearRegression().fit(X[manter], y[manter])
    return modelo

g = np.random.default_rng(3)
x = g.uniform(0, 10, 200)
y_sujo = 2 * x + 1 + g.normal(0, 1, 200)                       # verdade: intercepto 1, inclinação 2
y_sujo[g.choice(200, size=12, replace=False)] += g.uniform(40, 80, 12)    # 6% de outliers
X = x.reshape(-1, 1)

ols = LinearRegression().fit(X, y_sujo)
lts = regressao_truncada(X, y_sujo)
print("verdade : intercepto = 1.00 | inclinação = 2.00")
print(f"OLS     : intercepto = {ols.intercept_:.2f} | inclinação = {ols.coef_[0]:.2f}")
print(f"LTS     : intercepto = {lts.intercept_:.2f} | inclinação = {lts.coef_[0]:.2f}")
assert abs(lts.intercept_ - 1) < abs(ols.intercept_ - 1)

Nos *ensembles*, o mesmo raciocínio vale para agregar as previsões: se dois entre dez modelos quebram, a **média truncada das previsões** resiste e ainda usa a maior parte dos modelos:

In [ ]:
previsoes = np.array([10.1, 9.8, 10.3, 9.9, 10.0, 10.2, 10.4, 9.7, 250.0, -90.0])      # 2 modelos quebraram
print(f"média das previsões            = {previsoes.mean():.2f}")
print(f"mediana das previsões          = {np.median(previsoes):.2f}")
print(f"truncada 20% das previsões     = {stats.trim_mean(previsoes, 0.20):.2f}")

### 9.4 · Deep Learning: agregação robusta de gradientes

> 🏭 **No trabalho.** No **aprendizado federado**, vários clientes enviam gradientes que o servidor agrega. Se alguns clientes são **defeituosos ou maliciosos** (ataques bizantinos), a média simples é destruída. A agregação por **média truncada por coordenada** é uma defesa clássica:

In [ ]:
g = np.random.default_rng(4)
verdadeiro = np.array([1.0, -2.0, 0.5, 3.0, 0.0])
honestos = verdadeiro + g.normal(0, 0.3, size=(16, 5))
bizantinos = g.normal(0, 1, size=(4, 5)) * 100                  # 4 clientes enviam gradientes enormes
todos = np.vstack([honestos, bizantinos])

agregados = {
    "média":                    todos.mean(axis=0),
    "truncada 20% (por coord.)": stats.trim_mean(todos, 0.20, axis=0),
    "mediana":                  np.median(todos, axis=0),
}
erros = {nome: np.linalg.norm(v - verdadeiro) for nome, v in agregados.items()}
for nome, e in erros.items():
    print(f"{nome:<26} erro em relação ao gradiente verdadeiro = {e:8.3f}")
assert erros["truncada 20% (por coord.)"] < erros["média"]

Com 20% de clientes maliciosos, a truncada de 20% descarta até 4 valores por ponta em **cada coordenada** e recupera o gradiente. A mediana também resiste (neste experimento as duas ficaram próximas): ela usa só o valor central de cada coordenada, enquanto a truncada aproveita a maioria dos clientes honestos, o que costuma dar mais precisão quando o ruído deles é grande.

### 9.5 · Engenharia e Indústria: o tempo de ciclo padrão

> 🏭 **No trabalho (engenharia de produção).** O **tempo de ciclo** de uma máquina é medido várias vezes, mas algumas medições incluem **paradas** (troca de ferramenta, falta de material) que não fazem parte do ciclo normal. A média truncada estima o ciclo "típico" sem exigir que alguém classifique cada parada:

In [ ]:
g = np.random.default_rng(8)
tempos = g.normal(42, 1.5, 60)                                   # ciclo verdadeiro: 42 s
paradas = g.random(60) < 0.10
tempos[paradas] += g.uniform(30, 120, paradas.sum())             # ~10% das medições têm parada

for nome, valor in [("média", tempos.mean()), ("truncada 10%", stats.trim_mean(tempos, 0.10)),
                    ("truncada 20%", stats.trim_mean(tempos, 0.20)), ("mediana", np.median(tempos))]:
    print(f"{nome:<14} = {valor:6.2f} s   (erro de {abs(valor - 42):.2f} s)")
assert abs(stats.trim_mean(tempos, 0.20) - 42) < abs(tempos.mean() - 42)
print(f"\nmedições com parada: {paradas.sum()} de {len(tempos)}")

Repare que o corte precisa **cobrir a fração de paradas**: com cerca de 10% delas, uma truncada de 10% fica no limite, e uma de 20% tem folga. (O erro de 0,00 da mediana é coincidência desta amostra, e não uma vantagem sistemática.) Esse é exatamente o tema do mini-caso.

---
## 10 · Mini-caso: suavizando a vibração de um motor

**Contexto.** Um sensor de vibração mede o desgaste de um motor. O sinal verdadeiro tem uma **tendência crescente** (desgaste) e uma oscilação lenta, mas a leitura traz ruído e **picos espúrios** (de sinal aleatório) em uma fração $p$ das amostras. Queremos uma **média móvel truncada** que recupere o sinal. A pergunta é: **qual α usar, dado o quanto o sensor falha?**

A média móvel truncada usa, em cada instante, as $w$ leituras ao redor: ordena a janela, corta $k = \lfloor \alpha w \rfloor$ de cada ponta e tira a média do miolo.

In [ ]:
def simular_vibracao(p_pico, seed, n=800):
    g = np.random.default_rng(seed)
    t = np.arange(n)
    verdadeiro = 5 + 0.8 * np.sin(2 * np.pi * t / 400) + 0.003 * t         # oscilação lenta + tendência de desgaste
    obs = verdadeiro + g.normal(0, 0.5, n)
    picos = g.random(n) < p_pico
    obs[picos] += g.choice([-1, 1], picos.sum()) * g.uniform(5, 15, picos.sum())
    return verdadeiro, obs


def media_movel_truncada(obs, w, alfas):
    """Média móvel truncada para vários α de uma vez (ordena cada janela uma única vez)."""
    janelas = np.sort(sliding_window_view(np.pad(obs, w // 2, mode="reflect"), w), axis=1)
    return {a_: janelas[:, int(a_ * w):w - int(a_ * w)].mean(axis=1) for a_ in alfas}


w = 21
alfas_mc = [0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.49]
contaminacoes = [0, 0.02, 0.05, 0.10, 0.20]
sementes = range(10)

rmse = pd.DataFrame(index=[f"{p:.0%}" for p in contaminacoes], columns=[f"{a_:.0%}" for a_ in alfas_mc], dtype=float)
for p in contaminacoes:
    acumulado = np.zeros(len(alfas_mc))
    for s in sementes:
        verdadeiro, obs = simular_vibracao(p, seed=s)
        estimativas = media_movel_truncada(obs, w, alfas_mc)
        acumulado += [np.sqrt(np.mean((estimativas[a_] - verdadeiro) ** 2)) for a_ in alfas_mc]
    rmse.loc[f"{p:.0%}"] = acumulado / len(sementes)
rmse.index.name = "picos (p)"; rmse.columns.name = "corte α"

melhor = rmse.idxmin(axis=1)
display(rmse.round(3))
print("Melhor α para cada nível de contaminação:")
print(melhor.to_string())

valores_melhor = [float(v.strip("%")) / 100 for v in melhor]
assert melhor.iloc[0] in ("0%", "5%")                           # sem picos, quase não vale a pena cortar
assert valores_melhor[-1] >= 0.20                                # com 20% de picos, é preciso cortar bastante
assert all(b >= a for a, b in zip(valores_melhor, valores_melhor[1:]))   # o melhor α cresce com a contaminação

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.4), gridspec_kw={"width_ratios": [1, 1.4]})

# (a) mapa de calor: RMSE relativo ao melhor α de cada linha
relativo = rmse.div(rmse.min(axis=1), axis=0)
im = axs[0].imshow(relativo.values, cmap="viridis_r", aspect="auto")
axs[0].set_xticks(range(len(rmse.columns))); axs[0].set_xticklabels(rmse.columns)
axs[0].set_yticks(range(len(rmse.index))); axs[0].set_yticklabels(rmse.index)
for i in range(relativo.shape[0]):
    for j in range(relativo.shape[1]):
        axs[0].text(j, i, f"{relativo.iloc[i, j]:.1f}", ha="center", va="center",
                    color="white" if relativo.iloc[i, j] > 3 else "black", fontsize=8)
axs[0].grid(False); axs[0].set_xlabel("corte α"); axs[0].set_ylabel("fração de picos (p)")
axs[0].set_title("Erro relativo ao melhor α (1,0 = ótimo)")

# (b) um caso concreto: p = 10%
verdadeiro, obs = simular_vibracao(0.10, seed=0)
est = media_movel_truncada(obs, w, [0, 0.20, 0.49])
t = np.arange(len(obs)); zoom = slice(100, 300)
axs[1].plot(t[zoom], obs[zoom], ".", color="#aaaaaa", ms=4, label="leitura (picos fora do eixo foram cortados)")
axs[1].plot(t[zoom], verdadeiro[zoom], color="black", lw=1.6, label="sinal verdadeiro")
axs[1].plot(t[zoom], est[0][zoom], color=COR["media"], lw=1.8, label="média móvel (α = 0)")
axs[1].plot(t[zoom], est[0.49][zoom], color=COR["mediana"], lw=1.8, label="mediana móvel (α = 0,49)")
axs[1].plot(t[zoom], est[0.20][zoom], color=COR["trunc"], lw=2.4, label="truncada móvel (α = 0,20)")
axs[1].set_ylim(3, 9); axs[1].set_xlabel("amostra"); axs[1].set_ylabel("vibração")
axs[1].set_title("Caso p = 10%: quem recupera o sinal?"); axs[1].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

Três leituras do resultado:

1. **Sem picos, a média simples é a melhor**: cortar só joga informação fora. Cada pico a mais **empurra** o melhor α para cima.
2. O melhor α fica **bem acima** da taxa de picos (neste experimento, de 2 a 5 vezes). A razão é que a fração de picos **varia de janela para janela**: algumas janelas têm bem mais picos que a média e precisam de mais corte. Repare também que o erro é **assimétrico** no mapa de calor: cortar **de menos** custa caro (as células à esquerda do ótimo ficam bem acima de 1,0), e cortar **demais** custa pouco (à direita ficam perto de 1,0). Logo, é melhor errar **para cima**.
3. A **mediana móvel** (α próximo de 0,5) é a mais robusta, mas **desperdiça** os dados bons quando a contaminação é moderada. A truncada, com o α certo, tem o **melhor dos dois mundos**: elimina os picos e ainda suaviza com a maior parte das leituras.

> 🏭 **Regra prática.** Antes de escolher α, **meça** a taxa de picos (por exemplo, com uma amostra rotulada do sensor, como no filtro de Hampel da aula de Mediana) e comece com um α de **2 a 5 vezes essa taxa** (limitado a cerca de 0,4). Valide o α com um experimento como este, em vez de adotar "10%" por hábito.

---
# ATO VI · FIXAR
## 11 · Entrevista e exercícios

### 🎯 Perguntas de entrevista

Tente responder em voz alta antes de abrir cada resposta.

<details><summary><b>1. O que é a média truncada e quando você a usa?</b></summary>

É a média depois de descartar uma fração $\alpha$ dos valores de **cada** ponta. Usa-se quando há outliers ou caudas pesadas mas se quer aproveitar a maior parte dos dados, ou seja, um **compromisso** entre a eficiência da média e a robustez da mediana.
</details>

<details><summary><b>2. Qual é o ponto de ruptura da média truncada?</b></summary>

É igual à fração de corte $\alpha$. Com 10% de corte por ponta, ela tolera até 10% de contaminação. A média tem ponto de ruptura 0% e a mediana, 50%.
</details>

<details><summary><b>3. Qual é a diferença entre média truncada e média winsorizada?</b></summary>

A truncada **descarta** os extremos e divide por $n - 2k$. A winsorizada os **substitui** pelo valor vizinho mantido e divide por $n$. A winsorização é a base da **variância** e do erro padrão da média truncada.
</details>

<details><summary><b>4. Como escolher o α?</b></summary>

Estime a fração de outliers esperada e escolha um α **bem maior** que ela (no experimento da seção 10, de 2 a 5 vezes), porque cortar de menos custa muito mais que cortar demais. Sem informação, 10% a 20% é um compromisso seguro: perde pouca eficiência em dados limpos e protege em dados contaminados. O α também pode ser escolhido por validação (*bootstrap*) minimizando o erro padrão ou o erro de previsão.
</details>

<details><summary><b>5. Por que o desvio padrão dos valores que sobraram não serve como erro padrão?</b></summary>

Porque o corte "arruma" o miolo e reduz sua variabilidade, subestimando a incerteza real. O correto é usar a variância da amostra **winsorizada** com o fator $1/(1-2\alpha)$ (Tukey–McLaughlin) ou um *bootstrap*.
</details>

<details><summary><b>6. A média truncada estima o mesmo que a média?</b></summary>

Só em distribuições **simétricas**. Em distribuições assimétricas ela estima outro parâmetro (a média truncada populacional $\mu_\alpha$), em geral menor que $\mu$ quando há cauda à direita. Por isso **não serve** para estimar **totais** (receita, custo).
</details>

### ✏️ Exercícios

Resolva **antes** de olhar o gabarito no apêndice.

**Básico**

- **B1.** Calcule, na mão, a média, a mediana e a truncada de 20% de `[14, 3, 12, 11, 13, 12, 40, 12, 10, 13]`.
- **B2.** Calcule `stats.trim_mean(r, a)` para $\alpha \in \{0, 0{,}1, 0{,}2, 0{,}3, 0{,}4\}$. A partir de qual α ela difere da mediana em menos de 5%?
- **B3.** Com $n = 9$, o que faz $\alpha = 0{,}10$? Qual é o menor $\alpha$ que corta ao menos um valor por ponta?

**Intermediário**

- **I1.** Escreva `trimmean_excel_para_scipy(p)` que converte o argumento `p` do Excel (fração total) no `alfa` por ponta do SciPy, e verifique a equivalência quando $p\,n/2$ é inteiro.
- **I2.** Use *bootstrap* para encontrar o α (entre 0, 5%, 10%, 20%, 30%, 40% e 49%) que **minimiza o erro padrão** da média truncada da renda. O resultado muda quanto os dados têm mais outliers?
- **I3.** Compare o intervalo de confiança de 95% da truncada de 10% da renda calculado pela fórmula de Tukey–McLaughlin com o intervalo percentil por *bootstrap*.

**Desafio**

- **D1.** Demonstre que a média truncada é equivariante a transformações afins e que seu ponto de ruptura é $\alpha$.
- **D2.** Implemente o **teste de Yuen** (duas amostras, com variâncias winsorizadas) e confira com `scipy.stats.ttest_ind(trim=...)`.

In [ ]:
# ✏️ Resolva aqui (B1, B2, B3, I1, I2, I3, D2)

---
## 12 · Resumo e ponte

### 🧾 Cheat-sheet

| | Média truncada |
|---|---|
| **Definição** | média do miolo, após descartar $k = \lfloor \alpha n \rfloor$ valores de **cada** ponta |
| **SciPy** | `stats.trim_mean(x, alfa)` · `mstats.trimmed_mean(x, limits=(a, b))` |
| **Casos extremos** | α = 0: média · α → 0,5: mediana |
| **Ponto de ruptura** | α |
| **Escolha do α** | 2 a 5 vezes a fração esperada de outliers; 10% a 20% sem informação |
| **Erro padrão** | $s_w / ((1-2\alpha)\sqrt{n})$ (winsorizado) ou *bootstrap* |
| **Use quando** | valor **típico** com outliers e caudas pesadas |
| **Cuidado** | convenção de α · amostra pequena · viés em totais · `NaN` · erro padrão ingênuo |

### 🔑 As 7 ideias para levar

1. A média truncada é um **botão α** que vai da **média** (α = 0) até a **mediana** (α → 0,5).
2. Seu **ponto de ruptura é α**: cada estimador quebra exatamente na sua fração de corte.
3. Com α entre 10% e 20%, ela perde **pouca eficiência** em dados limpos e **ganha muito** com contaminação.
4. **Escolha α** acima da fração de outliers esperada (cortar de menos custa mais que cortar demais) e valide por simulação ou *bootstrap*.
5. O erro padrão exige a variância **winsorizada**: o desvio do miolo subestima a incerteza.
6. Em dados assimétricos ela estima o **valor típico**, e não a média, então **não use para totais**.
7. A ideia de "cortar as pontas" aparece no **teste de Yuen**, na **regressão LTS**, na **agregação robusta de gradientes** e na **média móvel** de sensores.

### 📚 Referências

- BUSSAB, W. O.; MORETTIN, P. A. *Estatística Básica*. Saraiva.
- MORETTIN, P. A.; SINGER, J. M. *Estatística e Ciência de Dados*. LTC.
- WILCOX, R. R. *Introduction to Robust Estimation and Hypothesis Testing*. Academic Press.
- HUBER, P. J.; RONCHETTI, E. M. *Robust Statistics*. Wiley.
- TUKEY, J. W. "A survey of sampling from contaminated distributions". 1960.
- YUEN, K. K. "The two-sample trimmed t for unequal population variances". *Biometrika*, 1974.
- Documentação: [`scipy.stats.trim_mean`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.trim_mean.html) · [`scipy.stats.ttest_ind`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html) · [`scipy.stats.mstats.winsorize`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mstats.winsorize.html)

### 🌉 Ponte para a próxima aula

A média, a mediana e a média truncada formam uma família inteira de medidas de posição, mas todas elas têm uma exigência em comum: os dados precisam ser **números**, que se possa somar ou, no mínimo, ordenar. E se os dados forem **categorias**?

In [ ]:
cat = df["categoria_frequente"]
try:
    cat.mean()
except Exception as erro:
    print(f"média de categorias -> {type(erro).__name__}")
print("\nCategoria mais frequente:", cat.value_counts().idxmax())
print(cat.value_counts(normalize=True).round(3).to_string())

Para variáveis **nominais** só resta **contar**. A medida de posição que responde "qual é o valor mais frequente?" é a **moda**, tema da próxima aula.

---
# 📎 APÊNDICE · Gabarito comentado

> Só abra depois de tentar resolver sozinho. 😉

In [ ]:
# ── B1 ─────────────────────────────────────────────────────────────
dados_b1 = [14, 3, 12, 11, 13, 12, 40, 12, 10, 13]
ordenados_b1 = sorted(dados_b1)                                  # [3, 10, 11, 12, 12, 12, 13, 13, 14, 40]
miolo_b1 = ordenados_b1[2:-2]                                    # k = floor(0,2 x 10) = 2 por ponta
print(f"B1  ordenados = {ordenados_b1}")
print(f"    média = {np.mean(dados_b1):.2f} | mediana = {np.median(dados_b1):.1f} | truncada 20% = {np.mean(miolo_b1):.2f} (miolo {miolo_b1})")
assert np.isclose(np.mean(miolo_b1), stats.trim_mean(dados_b1, 0.2))

# ── B2 ─────────────────────────────────────────────────────────────
md_r = np.median(r)
print("\nB2")
primeiro = None
for a_ in [0, 0.1, 0.2, 0.3, 0.4]:
    v = stats.trim_mean(r, a_)
    dif = abs(v - md_r) / md_r
    if primeiro is None and dif < 0.05:
        primeiro = a_
    print(f"    α = {a_:.1f} -> {v:8.2f} (diferença para a mediana: {dif:.1%})")
print(f"    a primeira α com diferença < 5% é {primeiro}")

# ── B3 ─────────────────────────────────────────────────────────────
print(f"\nB3  n = 9, α = 0,10 -> k = {int(0.10 * 9)} (nada é cortado!) | menor α que corta 1 por ponta: 1/9 = {1 / 9:.3f} -> k = {int((1 / 9 + 1e-9) * 9)}")

In [ ]:
# ── I1 ─────────────────────────────────────────────────────────────
def trimmean_excel_para_scipy(p):
    """TRIMMEAN(x, p) do Excel corresponde a trim_mean(x, p / 2) do SciPy (quando p·n/2 é inteiro)."""
    return p / 2

x40 = rng.normal(100, 20, 40)
excel_p = 0.20                                                   # 0,20 × 40 / 2 = 4 valores por ponta (inteiro)
assert np.isclose(trimmean_excel(x40, excel_p), stats.trim_mean(x40, trimmean_excel_para_scipy(excel_p)))
print(f"I1  Excel(p = {excel_p}) = {trimmean_excel(x40, excel_p):.3f} | SciPy(α = {trimmean_excel_para_scipy(excel_p)}) = {stats.trim_mean(x40, trimmean_excel_para_scipy(excel_p)):.3f}")

# ── I2 ─────────────────────────────────────────────────────────────
g = np.random.default_rng(6)
reamostras_i2 = [g.choice(r, size=len(r), replace=True) for _ in range(1000)]
eps = {}
for a_ in [0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.49]:
    eps[f"{a_:.0%}"] = np.std([stats.trim_mean(v, a_) for v in reamostras_i2], ddof=1)
eps = pd.Series(eps, name="erro padrão (bootstrap)")
print(f"\nI2  α com menor erro padrão da renda: {eps.idxmin()}")
display(eps.round(1).to_frame())

# ── I3 ─────────────────────────────────────────────────────────────
ic_form = (tm - tcrit * ep_form, tm + tcrit * ep_form)
ic_boot = tuple(np.percentile(boot, [2.5, 97.5]))
print(f"\nI3  IC Tukey–McLaughlin = [{ic_form[0]:.0f}, {ic_form[1]:.0f}] (largura {ic_form[1] - ic_form[0]:.0f})")
print(f"    IC bootstrap        = [{ic_boot[0]:.0f}, {ic_boot[1]:.0f}] (largura {ic_boot[1] - ic_boot[0]:.0f})")

**D1 · Demonstração.** *Equivariância afim:* se $y_i = a x_i + b$ com $a > 0$, a ordem é preservada, $y_{(i)} = a x_{(i)} + b$, e os mesmos $k$ índices são cortados. Logo $\bar{y}_\alpha = \frac{1}{n-2k}\sum_{i=k+1}^{n-k}(a x_{(i)} + b) = a\,\bar{x}_\alpha + b$. Se $a < 0$, a ordem se inverte, mas o corte é **simétrico** (os mesmos $k$ de cada lado), então o resultado vale também. *Ponto de ruptura:* a média truncada ignora os $k = \lfloor \alpha n \rfloor$ maiores e os $k$ menores valores. Se no máximo $k$ observações forem arbitrariamente grandes, todas caem na ponta cortada e o resultado permanece limitado pelos valores restantes. Com $k + 1$ valores arbitrários, ao menos um permanece no miolo e leva a média a qualquer valor. Então a fração tolerada é $k/n \approx \alpha$. $\blacksquare$

In [ ]:
# ── D2 ─────────────────────────────────────────────────────────────
def yuen(a, b, alfa=0.20):
    def partes(x):
        x = np.sort(np.asarray(x, dtype=float))
        n = len(x); k = int(alfa * n); h = n - 2 * k
        w = x.copy(); w[:k] = x[k]; w[n - k:] = x[n - k - 1]               # amostra winsorizada
        d = (n - 1) * w.var(ddof=1) / (h * (h - 1))
        return x[k:n - k].mean(), d, h
    m1, d1, h1 = partes(a)
    m2, d2, h2 = partes(b)
    t_obs = (m1 - m2) / np.sqrt(d1 + d2)
    gl = (d1 + d2) ** 2 / (d1 ** 2 / (h1 - 1) + d2 ** 2 / (h2 - 1))
    return t_obs, gl, 2 * stats.t.sf(abs(t_obs), gl)

g = np.random.default_rng(12)
a_y, b_y = g.lognormal(0, 1, 80), g.lognormal(0.5, 1, 100)
t_meu, gl_meu, p_meu = yuen(a_y, b_y, 0.20)
ref = stats.ttest_ind(a_y, b_y, equal_var=False, trim=0.20)
print(f"D2  nosso Yuen   : t = {t_meu:.4f} | gl = {gl_meu:.2f} | p = {p_meu:.4f}")
print(f"    scipy (trim) : t = {ref.statistic:.4f} | gl = {ref.df:.2f} | p = {ref.pvalue:.4f}")
assert np.isclose(t_meu, ref.statistic) and np.isclose(p_meu, ref.pvalue)